In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [3]:
spark = SparkSession.builder.appName('EBAC Lab 30').getOrCreate()

In [4]:
df_video = spark.read.parquet("/content/videos-preparados.snappy.parquet")
df_comments = spark.read.parquet("/content/videos-comments-tratados.snappy.parquet")

In [5]:
df_video.createOrReplaceTempView("videos_temp")
df_comments.createOrReplaceTempView("comments_temp")

In [ ]:
join_video_comments = spark.sql("""
    SELECT v.*, c.Comment, c.Sentiment, c.Likes Comment
    FROM videos_temp v
    JOIN comments_temp c ON v.`Video ID` = c.`Video ID`
""")

join_video_comments.show()

In [14]:
#Etapas 1,2,3 e 4 usando repartition e coalesce:

#1
df_video_rep = spark.read.parquet('/content/videos-preparados.snappy.parquet').repartition(8, 'Video ID')

#2
df_comments_rep = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet').repartition(8, 'Video ID')

#3
df_video_rep.createOrReplaceTempView('videos_rep')
df_comments_rep.createOrReplaceTempView('comments_rep')

#4
join_video_comments_rep = spark.sql("""
    SELECT v.*, c.Comment, c.Sentiment, c.Likes Comment
    FROM videos_rep v
    JOIN comments_rep c ON v.`Video ID` = c.`Video ID`
""")

In [ ]:
join_video_comments.explain(True)
join_video_comments_rep.explain(True)

In [32]:
print("DF VIDEO")
df_video.show()
print("DF COMMENTS")
df_comments.show()

DF VIDEO
+--------------------+-----------+------------+----------------+------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|         Keyword| Likes|Comments|    Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+----------------+------+--------+---------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|         mukbang|378858|   18860| 17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|            news|  6379|    4853|   808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  

In [57]:
#Join e filter apenas com os dados necessários:

#Lê os arquivos .parquet para carregar os DataFrames equivalentes:
df_video = spark.read.parquet('/content/videos-preparados.snappy.parquet')
df_comments = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet')

#Cria duas tabelas temporarias para a facilitação das consultas SQL:
df_video.createOrReplaceTempView('videos_opt')
df_comments.createOrReplaceTempView('comments_opt')

#Organiza os videos em um Ranking de maior para menor, levando em consideração o sucesso do vídeo (baseado na quantidade de interações e visualizações)
join_video_comments_opt = spark.sql("""
    SELECT DISTINCT v.`Video ID`, v.Title, v.Likes, v.Interaction
    FROM videos_opt v
    JOIN comments_opt c
    ON v.`Video ID` = c.`Video ID`
    WHERE v.Likes > 0 AND v.Interaction > 0
    ORDER BY v.Interaction DESC
""")

#Explicação do JOIN:
# em SELECT DISTINCT v.`Video ID`, v.Title, v.Likes, v.Interaction Essas são as colunas que serão utilizadas no processod e JOIN
# em FROM videos_opt v Essa é a minha tabela de videos
# em JOIN comments_opt c Essa é a minha tabela de comentários
# em ON v.`Video ID` = c.`Video ID` É a condição utilizada para fazer a junção de minhas duas tabelas
# em WHERE v.Likes > 0 AND v.Interaction > 0 Este é o filtro que esta sendo aplicado para efetuar uma pesquisa
# e ORDER BY v.Interaction DESC É a forma em que estou ordenando meus dados na tabela que sera apresentada.

print("Ranking de videos virais na plataforma:")
join_video_comments_opt.show()

Ranking de videos virais na plataforma:
+-----------+--------------------+--------+-----------+
|   Video ID|               Title|   Likes|Interaction|
+-----------+--------------------+--------+-----------+
|gCYcHz2k5x0|Martin Garrix - A...|11025176| 1593623628|
|XXYlFuWEuKI|The Weeknd - Save...| 6823113|  922551152|
|qpgTC9MDx1o|Maroon 5 - Animal...| 5743875|  832346002|
|jJPMnTXl63E|Powfu - death bed...| 7786057|  532691631|
|yjmp8CoZBIo|One Direction - H...| 5400589|  440187490|
|Ct6BUPvE2sM|PIKOTARO - PPAP (...| 4144389|  429916936|
|mRD0-GxqHVo|Glass Animals - H...| 6177588|  384467871|
|Ha80ZaecGkQ|Young Money - Bed...| 1430457|  323492195|
|nhBorPm6JjQ|Rihanna - Califor...| 1171433|  309733791|
|NvR60Wg9R7Q|Bon Jovi - Bed Of...| 1072689|  303235359|
|0e3GPea1Tyg|$456,000 Squid Ga...|14259033|  300397699|
|fyIcQ1Xl-rs|NLE Choppa - Walk...| 2286473|  253167182|
|EqboAI-Vk-U|Google — Year In ...|  169669|  239385460|
|nCg3ufihKyU|Tiësto - The Busi...| 1706185|  210025196|
|9bqk6ZU

In [53]:
#Salva meu JOIN otimizado em formato .parquet
join_video_comments_opt.write.mode('overwrite').parquet('join-videos-comments-otimizado')